# Aula 03 — Lab 02: Avaliação e Confiabilidade

**Validação estrutural, testes de prompts, regressão, retries, fallbacks e revisão humana**

Vamos continuar no cenário de resposta a incidentes, agora tratando o LLM como um componente
probabilístico que precisa de contratos, testes, telemetria e caminhos seguros de falha.

## Objetivos de aprendizagem

Ao final, você será capaz de:

1. exigir saída estruturada com Pydantic;
2. separar validação estrutural de regras de negócio;
3. criar um golden dataset de cenários representativos;
4. combinar avaliadores determinísticos e LLM-as-judge;
5. comparar versões de prompt e detectar regressões;
6. aplicar retry somente a falhas transitórias;
7. usar fallback e escalonamento humano com trilha de auditoria.

## Pirâmide de confiabilidade

Da base para o topo:

1. **contrato estrutural:** tipos, campos obrigatórios e limites;
2. **regras determinísticas:** políticas e invariantes verificáveis por código;
3. **avaliação semântica:** aderência, fundamentação e utilidade;
4. **observabilidade:** latência, custo, erros e versões;
5. **fallback e revisão humana:** saída segura quando a automação não é confiável.

Não use um segundo LLM para verificar algo que uma expressão booleana resolve melhor.

## 1. Instalação e configuração

Este notebook é independente do Lab 01. Instale o LangChain, Pydantic e a integração do
provedor escolhido.

In [ ]:
# Descomente e execute uma vez no ambiente do curso.
# %pip install -qU "langchain>=1.2,<2" "pydantic>=2,<3" python-dotenv pandas
# %pip install -qU langchain-anthropic
# %pip install -qU langchain-openai
# %pip install -qU langchain-google-genai
# %pip install -qU langchain-xai
# %pip install -qU langchain-deepseek

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

PROVIDERS = {
    "anthropic": ("anthropic", "ANTHROPIC_API_KEY", "ANTHROPIC_MODEL", "claude-sonnet-4-6"),
    "openai": ("openai", "OPENAI_API_KEY", "OPENAI_MODEL", "gpt-5.4-mini"),
    "google": ("google_genai", "GOOGLE_API_KEY", "GOOGLE_MODEL", "gemini-2.5-flash-lite"),
    "xai": ("xai", "XAI_API_KEY", "XAI_MODEL", "grok-4"),
    "deepseek": ("deepseek", "DEEPSEEK_API_KEY", "DEEPSEEK_MODEL", "deepseek-chat"),
}

requested_provider = os.getenv("LLM_PROVIDER", "").strip().lower()
if requested_provider and requested_provider not in PROVIDERS:
    raise ValueError(f"LLM_PROVIDER inválido: {requested_provider}")

provider = requested_provider or next(
    (name for name, values in PROVIDERS.items() if os.getenv(values[1])),
    "anthropic",
)
langchain_id, key_env, model_env, default_model = PROVIDERS[provider]
model_name = os.getenv(model_env, default_model)

if os.getenv(key_env):
    llm = init_chat_model(
        model=model_name,
        model_provider=langchain_id,
        temperature=0,
        max_tokens=1400,
        timeout=60,
        max_retries=0,
    )
    EXECUTAR_LLM = True
    print(f"Provedor: {provider} | modelo: {model_name}")
else:
    llm = None
    EXECUTAR_LLM = False
    print(f"Defina {key_env} no .env para executar chamadas ao LLM.")

## 2. Contrato de saída com Pydantic

Saída estruturada reduz ambiguidades e falha cedo quando tipos ou limites são violados.
Pydantic valida a **forma** da resposta. Regras como “todo P1 exige revisão humana” continuam
sendo responsabilidade da aplicação.

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field

class AnaliseIncidente(BaseModel):
    "Resultado estruturado da triagem de um incidente."

    categoria: Literal[
        "api", "banco", "seguranca", "fila", "rede", "atendimento", "desconhecido"
    ]
    severidade: Literal["P1", "P2", "P3", "P4"]
    resumo: str = Field(min_length=10, max_length=300)
    acao_recomendada: str = Field(min_length=10, max_length=500)
    evidencias: List[str] = Field(
        default_factory=list,
        description="IDs das fontes usadas; vazia somente quando nenhuma fonte é aplicável",
    )
    confianca: float = Field(ge=0.0, le=1.0)
    precisa_revisao_humana: bool

print(AnaliseIncidente.model_json_schema())

## 3. Duas versões de prompt

A versão A é uma baseline curta. A versão B contém critérios observáveis, política de
incerteza e regras de escalonamento. Não assumiremos que B é melhor: vamos medir.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

PROMPTS = {
    "A": ChatPromptTemplate.from_messages([
        ("system", "Classifique o incidente e recomende uma ação."),
        ("human", "INCIDENTE: {incidente}\nFONTES DISPONÍVEIS: {fontes}"),
    ]),
    "B": ChatPromptTemplate.from_messages([
        ("system", '''Você faz triagem de incidentes da SkillGo.

OBJETIVO
Produzir uma classificação conservadora e verificável.

REGRAS
- Use apenas o incidente e as fontes disponibilizadas.
- Evidencias deve conter somente IDs presentes em FONTES DISPONÍVEIS.
- Evidencias pode ser vazia somente nas categorias atendimento ou desconhecido.
- Não invente causa raiz, métrica, comando ou fonte.
- Use desconhecido quando os dados não sustentarem uma categoria.
- P1 e segurança sempre exigem revisão humana.
- Confiança abaixo de 0,70 exige revisão humana.
- A ação recomendada deve ser reversível e não destrutiva.

CRITÉRIO DE PARADA
Se faltarem evidências, classifique conservadoramente e solicite revisão.'''),
        ("human", "INCIDENTE\n{incidente}\n\nFONTES DISPONÍVEIS\n{fontes}"),
    ]),
}

FONTES_PADRAO = "RB-API-01, RB-DB-01, RB-AUTH-01, RB-QUEUE-01, RB-DNS-01, POL-COM-01"

In [ ]:
from time import perf_counter

def analisar_incidente(incidente, versao="B", fontes=FONTES_PADRAO):
    if llm is None:
        raise RuntimeError("Configure uma chave de API antes da análise.")

    modelo_estruturado = llm.with_structured_output(
        AnaliseIncidente,
        include_raw=True,
    )
    chain = PROMPTS[versao] | modelo_estruturado
    inicio = perf_counter()
    resultado = chain.invoke({"incidente": incidente, "fontes": fontes})

    return {
        "parsed": resultado["parsed"],
        "parsing_error": resultado["parsing_error"],
        "raw": resultado["raw"],
        "latency_s": perf_counter() - inicio,
        "prompt_version": versao,
        "model": model_name,
    }

incidente_exemplo = (
    "Após a rotação de credenciais às 14:00, respostas 401 subiram de 0,2% para 18%. "
    "O identificador da chave em dois pods ainda é o anterior."
)

if EXECUTAR_LLM:
    exemplo = analisar_incidente(incidente_exemplo)
    print(exemplo["parsed"].model_dump())
    print("Erro de parsing:", exemplo["parsing_error"])
    print("Latência:", round(exemplo["latency_s"], 2), "s")
else:
    print("Configure uma chave para executar o exemplo.")

## 4. Validação estrutural versus validação semântica

A estrutura pode estar perfeita e ainda violar uma regra do negócio. O validador abaixo é
determinístico, barato e reproduzível.

In [ ]:
import re

FONTES_VALIDAS = set(FONTES_PADRAO.split(", "))

def validar_regras(analise, fontes_validas=FONTES_VALIDAS):
    erros = []

    fontes_inventadas = set(analise.evidencias) - set(fontes_validas)
    if fontes_inventadas:
        erros.append(f"Fontes não autorizadas: {sorted(fontes_inventadas)}")

    if not analise.evidencias and analise.categoria not in {"atendimento", "desconhecido"}:
        erros.append("Categoria operacional exige pelo menos uma evidência.")

    if analise.severidade == "P1" and not analise.precisa_revisao_humana:
        erros.append("Todo P1 exige revisão humana.")

    if analise.categoria == "seguranca" and not analise.precisa_revisao_humana:
        erros.append("Incidentes de segurança exigem revisão humana.")

    if analise.confianca < 0.70 and not analise.precisa_revisao_humana:
        erros.append("Baixa confiança exige revisão humana.")

    padroes_destrutivos = r"\b(apagar|deletar|drop|reiniciar banco|formatar)\b"
    if re.search(padroes_destrutivos, analise.acao_recomendada.lower()):
        erros.append("Ação potencialmente destrutiva recomendada.")

    return {"valid": not erros, "errors": erros}

saida_teste = AnaliseIncidente(
    categoria="seguranca",
    severidade="P1",
    resumo="Falhas 401 aumentaram após rotação de credencial.",
    acao_recomendada="Comparar o identificador da chave nos pods afetados.",
    evidencias=["RB-AUTH-01"],
    confianca=0.86,
    precisa_revisao_humana=False,
)

print(validar_regras(saida_teste))

## 5. Golden dataset

Um conjunto dourado pequeno deve conter casos comuns, limites, ambiguidades e situações de
alto risco. Comece com poucos casos revisados por especialistas e acrescente falhas reais de
produção ao longo do tempo.

In [ ]:
GOLDEN_DATASET = [
    {
        "id": "INC-001",
        "input": "Pool de conexões em 96%, 120 requisições aguardando e timeouts no checkout.",
        "expected_category": "banco",
        "expected_severity": "P1",
        "must_review": True,
        "allowed_sources": {"RB-DB-01", "POL-COM-01"},
    },
    {
        "id": "INC-002",
        "input": "Latência p95 da API subiu após deploy, sem aumento de erros.",
        "expected_category": "api",
        "expected_severity": "P2",
        "must_review": False,
        "allowed_sources": {"RB-API-01"},
    },
    {
        "id": "INC-003",
        "input": "Respostas 401 aumentaram após rotação; dois pods usam a chave anterior.",
        "expected_category": "seguranca",
        "expected_severity": "P1",
        "must_review": True,
        "allowed_sources": {"RB-AUTH-01", "POL-COM-01"},
    },
    {
        "id": "INC-004",
        "input": "Backlog da fila cresce 2 mil mensagens por minuto; consumidores estão saudáveis.",
        "expected_category": "fila",
        "expected_severity": "P2",
        "must_review": False,
        "allowed_sources": {"RB-QUEUE-01"},
    },
    {
        "id": "INC-005",
        "input": "Um usuário não recebeu o e-mail de boas-vindas. Não há outros relatos.",
        "expected_category": "atendimento",
        "expected_severity": "P4",
        "must_review": False,
        "allowed_sources": set(),
    },
    {
        "id": "INC-006",
        "input": "O sistema parece estranho desde ontem. Nenhuma métrica foi fornecida.",
        "expected_category": "desconhecido",
        "expected_severity": "P3",
        "must_review": True,
        "allowed_sources": set(),
    },
]

print(f"{len(GOLDEN_DATASET)} cenários no golden dataset")

## 6. Avaliadores determinísticos

Cada métrica responde a uma pergunta específica. Uma média única pode esconder uma regressão
crítica, por isso também manteremos métricas de segurança como gates obrigatórios.

In [ ]:
def avaliar_caso(analise, caso):
    regras = validar_regras(analise)
    fontes = set(analise.evidencias)
    fontes_permitidas = caso["allowed_sources"]

    if fontes_permitidas:
        citacoes_validas = fontes.issubset(fontes_permitidas) and bool(fontes)
    else:
        citacoes_validas = not fontes

    return {
        "id": caso["id"],
        "categoria_correta": analise.categoria == caso["expected_category"],
        "severidade_correta": analise.severidade == caso["expected_severity"],
        "revisao_correta": analise.precisa_revisao_humana == caso["must_review"],
        "citacoes_validas": citacoes_validas,
        "regras_validas": regras["valid"],
        "erros_regra": regras["errors"],
    }

def agregar_metricas(avaliacoes):
    campos = [
        "categoria_correta",
        "severidade_correta",
        "revisao_correta",
        "citacoes_validas",
        "regras_validas",
    ]
    return {
        campo: sum(bool(item[campo]) for item in avaliacoes) / len(avaliacoes)
        for campo in campos
    }

## 7. Testando o sistema de avaliação sem gastar tokens

Antes de avaliar o LLM, teste os próprios avaliadores com saídas controladas. A versão candidata
abaixo contém regressões propositais: severidade errada e ausência indevida de revisão humana.

In [ ]:
def criar_saida(caso, versao):
    categoria = caso["expected_category"]
    severidade = caso["expected_severity"]
    revisao = caso["must_review"]
    fontes = sorted(caso["allowed_sources"])[0:1]

    if versao == "candidata" and caso["id"] == "INC-003":
        severidade = "P2"
        revisao = False

    return AnaliseIncidente(
        categoria=categoria,
        severidade=severidade,
        resumo=f"Triagem controlada para o cenário {caso['id']}.",
        acao_recomendada="Coletar evidências adicionais e seguir o runbook aplicável.",
        evidencias=fontes,
        confianca=0.85,
        precisa_revisao_humana=revisao,
    )

resultados_simulados = {}
for versao in ["baseline", "candidata"]:
    avaliacoes = [
        avaliar_caso(criar_saida(caso, versao), caso)
        for caso in GOLDEN_DATASET
    ]
    resultados_simulados[versao] = agregar_metricas(avaliacoes)

resultados_simulados

### Nota didática sobre contrato e ausência de evidência

O schema permite lista vazia porque existem casos sem runbook aplicável. A regra de negócio
restringe essa exceção às categorias atendimento e desconhecido. Assim, tipos validam a forma
geral e código determinístico valida a relação entre campos.

In [ ]:
import pandas as pd

tabela_metricas = pd.DataFrame(resultados_simulados).T
display(tabela_metricas.style.format("{:.0%}"))

def quality_gate(metricas, limites):
    falhas = {
        metrica: {"observado": metricas.get(metrica, 0), "mínimo": minimo}
        for metrica, minimo in limites.items()
        if metricas.get(metrica, 0) < minimo
    }
    return {"approved": not falhas, "failures": falhas}

LIMITES = {
    "categoria_correta": 0.90,
    "severidade_correta": 0.90,
    "revisao_correta": 1.00,
    "citacoes_validas": 1.00,
    "regras_validas": 1.00,
}

print("Baseline:", quality_gate(resultados_simulados["baseline"], LIMITES))
print("Candidata:", quality_gate(resultados_simulados["candidata"], LIMITES))

## 8. Testes de contrato do prompt

Estes testes não chamam o modelo. Eles detectam remoção acidental de variáveis, regras e
critérios essenciais durante uma refatoração.

In [ ]:
def texto_do_prompt(prompt):
    return " ".join(str(message.prompt.template) for message in prompt.messages)

def testar_contrato_prompt(prompt):
    texto = texto_do_prompt(prompt).lower()
    variaveis = set(prompt.input_variables)

    assert {"incidente", "fontes"}.issubset(variaveis)
    assert "não invente" in texto
    assert "revisão humana" in texto
    assert "confiança" in texto
    assert "fontes disponibilizadas" in texto
    return "Contrato do prompt aprovado."

print(testar_contrato_prompt(PROMPTS["B"]))

## 9. Avaliação real das versões A e B

A célula abaixo pode executar até doze chamadas. Ela vem desabilitada para evitar custo
acidental. Os resultados incluem erros de parsing, latência, uso de tokens e métricas por caso.

In [ ]:
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from typing import Any, Dict, Optional

@dataclass
class RegistroAvaliacao:
    case_id: str
    prompt_version: str
    model: str
    timestamp: str
    latency_s: float
    input_tokens: Optional[int]
    output_tokens: Optional[int]
    parsed: bool
    error: Optional[str]
    metrics: Dict[str, Any]

def executar_suite(versao, dataset=GOLDEN_DATASET):
    registros = []

    for caso in dataset:
        inicio = perf_counter()
        try:
            resultado = analisar_incidente(
                caso["input"],
                versao=versao,
                fontes=", ".join(sorted(caso["allowed_sources"])) or "NENHUMA FONTE APLICÁVEL",
            )
            analise = resultado["parsed"]
            if analise is None:
                raise ValueError(str(resultado["parsing_error"]))

            metricas = avaliar_caso(analise, caso)
            usage = resultado["raw"].usage_metadata or {}
            erro = None
        except Exception as exc:
            metricas = {
                "categoria_correta": False,
                "severidade_correta": False,
                "revisao_correta": False,
                "citacoes_validas": False,
                "regras_validas": False,
            }
            usage = {}
            erro = f"{type(exc).__name__}: {exc}"

        registros.append(RegistroAvaliacao(
            case_id=caso["id"],
            prompt_version=versao,
            model=model_name,
            timestamp=datetime.now(timezone.utc).isoformat(),
            latency_s=round(perf_counter() - inicio, 3),
            input_tokens=usage.get("input_tokens"),
            output_tokens=usage.get("output_tokens"),
            parsed=erro is None,
            error=erro,
            metrics=metricas,
        ))

    return registros

EXECUTAR_SUITE_REAL = False

if EXECUTAR_SUITE_REAL and EXECUTAR_LLM:
    registros_reais = executar_suite("A") + executar_suite("B")
    df_real = pd.DataFrame([asdict(r) for r in registros_reais])
    display(df_real)
else:
    print("Defina EXECUTAR_SUITE_REAL=True para executar a avaliação com LLM.")

## 10. Avaliação semântica com LLM-as-judge

Use um juiz apenas para qualidades difíceis de codificar, como clareza ou utilidade. O juiz
também é probabilístico: use critérios explícitos, amostras revisadas por humanos e, quando
necessário, repetições ou comparação pareada.

In [ ]:
class Julgamento(BaseModel):
    nota: int = Field(ge=1, le=5)
    fundamentada_nas_evidencias: bool
    acao_segura: bool
    justificativa: str = Field(min_length=10, max_length=500)

JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", '''Você avalia uma resposta de triagem.
Dê nota de 1 a 5 considerando: aderência às evidências, utilidade e segurança.
Não recompense texto longo. Penalize fatos não sustentados e ações destrutivas.'''),
    ("human", '''INCIDENTE
{incidente}

RESPOSTA
{resposta}

REFERÊNCIA
{referencia}'''),
])

def avaliar_com_juiz(incidente, resposta, referencia):
    if llm is None:
        raise RuntimeError("Configure uma chave de API para usar o juiz.")
    judge = llm.with_structured_output(Julgamento)
    return (JUDGE_PROMPT | judge).invoke({
        "incidente": incidente,
        "resposta": resposta,
        "referencia": referencia,
    })

USAR_LLM_JUIZ = False
print("Juiz configurado. Ative USAR_LLM_JUIZ somente em uma amostra dos casos.")

## 11. Retry e fallback

**Retry** é indicado para timeout, limite temporário ou indisponibilidade. Repetir a mesma
chamada após uma violação de regra tende a repetir o erro; nesse caso, forneça feedback,
altere a estratégia ou escale.

**Fallback** troca de modelo, provedor ou estratégia após uma falha. O fallback deve preservar
o mesmo contrato de entrada e saída.

In [ ]:
from langchain_core.runnables import RunnableLambda

contador = {"tentativas": 0}

def servico_instavel(valor):
    contador["tentativas"] += 1
    if contador["tentativas"] < 3:
        raise TimeoutError("Falha transitória simulada")
    return {"status": "ok", "valor": valor}

retry_demo = RunnableLambda(servico_instavel).with_retry(
    retry_if_exception_type=(TimeoutError,),
    stop_after_attempt=3,
    wait_exponential_jitter=False,
)

print(retry_demo.invoke("INC-001"))
print("Tentativas:", contador["tentativas"])

In [ ]:
def primario_indisponivel(valor):
    raise ConnectionError("Provedor primário indisponível")

def resposta_degradada(valor):
    return {
        "status": "degradado",
        "mensagem": "Não foi possível automatizar a análise.",
        "acao": "Encaminhar para revisão humana.",
        "input": valor,
    }

fallback_demo = RunnableLambda(primario_indisponivel).with_fallbacks([
    RunnableLambda(resposta_degradada)
])

print(fallback_demo.invoke("INC-002"))

### Fallback real entre provedores

Quando houver um segundo modelo configurado, a mesma ideia pode ser aplicada ao Runnable do
modelo. Garanta que ambos suportem a estratégia de saída estruturada escolhida e registre qual
provedor produziu a resposta.

In [ ]:
def criar_fallback_real():
    fallback_provider = os.getenv("FALLBACK_PROVIDER", "").strip().lower()
    if not fallback_provider:
        return llm
    if fallback_provider not in PROVIDERS:
        raise ValueError(f"FALLBACK_PROVIDER inválido: {fallback_provider}")

    fallback_lc_id, fallback_key, fallback_model_env, fallback_default = PROVIDERS[fallback_provider]
    if not os.getenv(fallback_key):
        raise RuntimeError(f"Defina {fallback_key} para ativar o fallback.")

    fallback_model = init_chat_model(
        model=os.getenv(fallback_model_env, fallback_default),
        model_provider=fallback_lc_id,
        temperature=0,
        max_tokens=1400,
        timeout=60,
        max_retries=0,
    )

    primario_com_retry = llm.with_retry(
        retry_if_exception_type=(TimeoutError, ConnectionError),
        stop_after_attempt=2,
    )
    return primario_com_retry.with_fallbacks([fallback_model])

if EXECUTAR_LLM:
    llm_resiliente = criar_fallback_real()
    print("Runnable resiliente configurado.")
else:
    llm_resiliente = None
    print("Configure o modelo primário antes do fallback real.")

## 12. Política de escalonamento para revisão humana

A revisão humana não é apenas um último recurso técnico. Ela deve ser acionada por risco,
incerteza ou conflito de política, mesmo quando a chamada ao modelo foi bem-sucedida.

In [ ]:
def decidir_destino(analise, validacao, parsing_error=None):
    motivos = []

    if parsing_error:
        motivos.append("saída estrutural inválida")
    if not validacao["valid"]:
        motivos.extend(validacao["errors"])
    if analise is None:
        return {"route": "human_review", "reasons": motivos or ["sem análise"]}
    if analise.severidade == "P1":
        motivos.append("incidente P1")
    if analise.categoria == "seguranca":
        motivos.append("incidente de segurança")
    if analise.confianca < 0.70:
        motivos.append("confiança abaixo de 0,70")
    if analise.precisa_revisao_humana:
        motivos.append("modelo solicitou revisão")

    return {
        "route": "human_review" if motivos else "automatic_response",
        "reasons": sorted(set(motivos)),
    }

exemplo_validacao = validar_regras(saida_teste)
print(decidir_destino(saida_teste, exemplo_validacao))

In [ ]:
from uuid import uuid4

fila_revisao = []

def escalar_para_humano(incidente, analise, motivos, historico_tentativas):
    item = {
        "review_id": str(uuid4()),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "incident": incidente,
        "analysis": analise.model_dump() if analise else None,
        "reasons": motivos,
        "attempt_history": historico_tentativas,
        "status": "pending",
    }
    fila_revisao.append(item)
    return item

decisao = decidir_destino(saida_teste, exemplo_validacao)
if decisao["route"] == "human_review":
    revisao = escalar_para_humano(
        incidente=incidente_exemplo,
        analise=saida_teste,
        motivos=decisao["reasons"],
        historico_tentativas=[{"attempt": 1, "model": model_name, "status": "policy_failure"}],
    )
    print(revisao)

## 13. Pipeline confiável: visão integrada

1. validar entrada e permissões;
2. selecionar somente o contexto necessário;
3. executar o modelo com timeout e telemetria;
4. validar schema;
5. verificar regras de negócio e fundamentação;
6. aplicar retry apenas em falhas transitórias;
7. aplicar fallback compatível;
8. escalar por risco, baixa confiança ou falha persistente;
9. registrar versão do prompt, modelo, contexto, métricas e decisão humana;
10. transformar falhas reais em novos casos de regressão.

## 14. Checklist para produção

- Há schema e limites para toda saída consumida por software?
- Regras críticas são verificadas por código?
- O golden dataset inclui casos comuns, limites e alto risco?
- Métricas possuem limites mínimos e gates não negociáveis?
- Prompt, modelo, documentos e avaliadores são versionados?
- Retries são limitados e restritos a erros transitórios?
- Fallback preserva contrato e não mascara falhas silenciosamente?
- Revisão humana recebe evidências e histórico suficientes?
- Dados sensíveis são removidos de logs, prompts e datasets?
- Falhas de produção retornam para a suíte de regressão?

## Exercícios

1. Corrija o contrato de evidências para representar explicitamente “sem fonte aplicável”.
2. Adicione casos adversariais: prompt injection, fonte inexistente e ação destrutiva.
3. Crie uma métrica de custo por caso usando usage_metadata.
4. Compare as versões A e B com três repetições por cenário.
5. Configure FALLBACK_PROVIDER e provoque uma falha do modelo primário.
6. Implemente estados approve, edit e reject na fila de revisão humana.
7. Exporte os casos que falharam para um novo dataset de regressão.

## Referências

- [LangChain — Structured output](https://docs.langchain.com/oss/python/langchain/structured-output)
- [LangChain — Models and structured output](https://docs.langchain.com/oss/python/langchain/models)
- [LangChain — Guardrails](https://docs.langchain.com/oss/python/langchain/guardrails)
- [LangChain — Human in the loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- [LangSmith — Evaluation](https://docs.langchain.com/langsmith/evaluation)
- [LangSmith — Local evaluation](https://docs.langchain.com/langsmith/local)

**Mensagem final:** confiabilidade não vem de um prompt perfeito. Ela nasce de contratos,
contexto controlado, avaliações contínuas e caminhos seguros quando o modelo falha.